# MLP Analysis — CIC-IDS-2018 (Pucktrick Data Quality)

Esperimenti di noise injection sul dataset **CIC-IDS-2018** (network intrusion detection)
utilizzando la libreria **Pucktrick** per corrompere controllabilmente le feature
e misurare l'impatto sulle performance di un MLP (Spark MLlib).

In [ ]:
# === STEP 1: Import e configurazione ===

LOCAL_RUN = False  # True = esecuzione locale, False = cluster Spark remoto

# Standard library
import time
import json
import os
import subprocess
from itertools import product

# Spark core
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructType, StructField

# Spark MLlib — feature engineering
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Spark MLlib — modello
from pyspark.ml.classification import MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel
# Spark MLlib — valutazione
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Spark MLlib — pipeline
from pyspark.ml import Pipeline
from pyspark.ml import PipelineModel

# Pucktrick
from pucktrick import PuckTrick, Engine

# Solo per export finale — NON usato nel loop sperimentale
import pandas as pd

# Paths
if LOCAL_RUN:
    PATH = "DATASETS"
    PATH_RESULTS = "RESULTS"
else:
    PATH = "/home/PuckTrickadmin/DATASETS/"  # senza file:// — lettura via pandas (locale al driver)
    PATH_RESULTS = "/home/PuckTrickadmin/cava/RESULTS"

## Step 2 — Configurazione Spark e caricamento dataset

In questa sezione inizializziamo la `SparkSession` (locale o su cluster remoto)
e carichiamo il dataset **CIC-IDS-2018** preprocessato (`all_elaborated.parquet`).

Il dataset proviene dall'analisi esplorativa condotta nel notebook `cavas_dataset_analysis.ipynb`,
dove sono state già rimosse:
- Feature a **varianza zero** (8 colonne costanti)
- Feature **altamente correlate** tra loro (|r| ≥ 0.95, 27 colonne ridondanti)
- Righe con **valori nulli** (< 0.4% del dataset)
- Righe con **timestamp corrotti** (epoch/1970)

Il preprocessing si limita a:
- **Cast** di tutte le feature numeriche a `DoubleType` (requisito Spark MLlib)
- **Sostituzione** di valori infiniti con `NaN` (presente in `Flow Byts/s` e `Flow Pkts/s`)
- **Split temporale** basato sul `Timestamp`

Infine, inizializziamo `PuckTrick` con backend Spark: questo aggiunge automaticamente
la colonna `_pucktrick_row_id` al DataFrame, necessaria per tracciare le righe
modificate durante l'iniezione del rumore. Il DataFrame risultante (`base_df`)
è la **base immutabile** da cui partono tutti gli esperimenti.

### Descrizione del dataset

| Proprietà | Valore |
|---|---|
| Dominio | Network Intrusion Detection (CIC-IDS-2018) |
| Periodo | 2018-02-14 → 2018-03-02 (10 giornate) |
| Target binario | `Label_generic` (0 = Benign, 1 = Malign) |
| Target multi-classe | `Label` (tipo di attacco specifico) |
| Feature dopo preprocessing | 43 numeriche |
| Sbilanciamento | ~83% Benign, ~17% Malign |

### Costanti sperimentali

| Parametro | Valore |
|---|---|
| Feature per noise injection | `Fwd Seg Size Min`, `Bwd Pkts/s`, `Fwd Pkt Len Mean` |
| Tipi di rumore | `duplicated`, `labels`, `missing`, `noisy`, `outliers` |
| Livelli di rumore | 0%, 10%, 20%, 30%, 50% |
| Run per combinazione | 20 |
| t di Student (95%, df=19) | 2.093 |

### Scelta delle feature per il noise injection

Le 3 feature selezionate per l'iniezione di rumore sono le **più correlate
con `Label_generic`** (analisi dal notebook di esplorazione):

1. **`Fwd Seg Size Min`** — |corr| = 0.43, categorica (11 valori), 🟢 robusta
2. **`Bwd Pkts/s`** — |corr| = 0.26, continua, 🔴 alta fragilità
3. **`Fwd Pkt Len Mean`** — |corr| = 0.23, continua, 🔴 alta fragilità

Questa selezione copre profili diversi di fragilità al rumore: una feature
robusta ma altamente informativa, e due fragili dove ci aspettiamo degradazione
più marcata.

### Nota statistica — Intervallo di Confidenza al 95%

Per ogni combinazione *(tipo di rumore × feature × percentuale)* eseguiamo **20 run indipendenti**,
ciascuno con un seed diverso. Questo ci permette di stimare non solo la metrica media,
ma anche la sua **variabilità** tramite l'intervallo di confidenza al 95%.

La formula utilizzata è quella dell'intervallo di confidenza per campioni piccoli (distribuzione t di Student):

$$IC = \bar{x} \pm t \cdot \frac{s}{\sqrt{n}}$$

Dove:
- $\bar{x}$ è la **media** dell'F1-score sui 20 run
- $s$ è la **deviazione standard** campionaria
- $n = 20$ è il numero di run
- $t = 2.093$ è il valore critico dalla **tavola t di Student** con $df = n - 1 = 19$ gradi di libertà e livello di confidenza al 95%

**Esempio pratico:** se sui 20 run otteniamo F1 medio $= 0.85$ con deviazione standard $= 0.03$:

$$IC = 0.85 \pm 2.093 \cdot \frac{0.03}{\sqrt{20}} = 0.85 \pm 0.014 = [0.836,\ 0.864]$$

Questo significa che siamo al 95% confidenti che il vero valore dell'F1 — quello che otterremmo
con infiniti run — cada nell'intervallo $[0.836, 0.864]$.

Confrontando gli intervalli tra livelli di rumore diversi, possiamo stabilire se una degradazione
delle performance è **statisticamente significativa** o rientra nella variabilità naturale del modello.

> **Nota:** il valore $t = 2.093$ è valido **solo con $n = 20$ run**. Se si modifica `N_RUNS`,
> aggiornare `T_VALUE_95` consultando la tavola t di Student alla riga $df = N\_RUNS - 1$.

Link alla tavola student
https://www.statisticshowto.com/tables/t-distribution-table/

In [ ]:
# === STEP 2: Configurazione Spark e caricamento dataset CIC-IDS-2018 ===

# ── Configurazione Spark ───────────────────────────────────────────────────
if LOCAL_RUN:
    # Trova Java automaticamente se necessario
    java_home = os.environ.get('JAVA_HOME', '')
    if not java_home:
        try:
            java_path = subprocess.check_output(['which', 'java'], text=True).strip()
            java_home = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))
            os.environ['JAVA_HOME'] = java_home
        except subprocess.CalledProcessError:
            print("⚠️ Java non trovato! Installare: sudo apt install default-jdk")

    os.environ['PYSPARK_PYTHON'] = 'python3'
    os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

    spark = SparkSession.builder \
        .appName("CICIDS_MLP_Experiments") \
        .master("local[*]") \
        .config("spark.driver.memory", "14g") \
        .config("spark.driver.host", "localhost") \
        .config("spark.ui.showConsoleProgress", "false") \
        .getOrCreate()
else:
    MASTER_URL  = "spark://10.0.1.8:7077"
    DRIVER_HOST = "10.0.1.8"

    spark = SparkSession.builder \
        .appName("CICIDS_MLP_Experiments") \
        .master(MASTER_URL) \
        .config("spark.submit.deployMode",      "client") \
        .config("spark.executor.instances",     "4") \
        .config("spark.executor.cores",         "4") \
        .config("spark.executor.memory",        "13g") \
        .config("spark.driver.memory",          "8g") \
        .config("spark.driver.host",            DRIVER_HOST) \
        .config("spark.driver.bindAddress",     DRIVER_HOST) \
        .config("spark.sql.shuffle.partitions", "32") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("ERROR")

print("SparkSession creata — versione:", spark.version)

# ── Costanti sperimentali ──────────────────────────────────────────────────
# Le 43 feature numeriche rimaste dopo il preprocessing (vedi cavas_dataset_analysis.ipynb)
ALL_FEATURES = [
    "Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts",
    "TotLen Fwd Pkts", "Fwd Pkt Len Max", "Fwd Pkt Len Min", "Fwd Pkt Len Mean",
    "Bwd Pkt Len Max", "Bwd Pkt Len Min", "Bwd Pkt Len Mean",
    "Flow Byts/s", "Flow Pkts/s",
    "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
    "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
    "Fwd PSH Flags", "Fwd URG Flags",
    "Fwd Pkts/s", "Bwd Pkts/s",
    "Pkt Len Min", "Pkt Len Mean", "Pkt Len Var",
    "FIN Flag Cnt", "RST Flag Cnt", "PSH Flag Cnt", "ACK Flag Cnt", "URG Flag Cnt",
    "Down/Up Ratio",
    "Init Fwd Win Byts", "Init Bwd Win Byts",
    "Fwd Seg Size Min",
    "Active Mean", "Active Std", "Active Max", "Active Min",
    "Idle Min",
]
N_FEATURES = len(ALL_FEATURES)

# Top-3 feature più correlate con Label_generic — target del noise injection
NOISE_FEATURES = ["Fwd Seg Size Min", "Bwd Pkts/s", "Fwd Pkt Len Mean"]

LABEL_COL    = "Label_generic"
NOISE_TYPES  = ["duplicated", "labels", "missing", "noisy", "outliers"]
NOISE_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.5]
N_RUNS       = 20
T_VALUE_95   = 2.093

print(f"Feature totali: {N_FEATURES}")
print(f"Feature per noise injection: {NOISE_FEATURES}")

# ── Caricamento dataset ────────────────────────────────────────────────────
# NOTA: in Spark standalone, file:/// è locale al nodo che esegue il task.
# Gli executor su 10.0.1.6 non possono accedere ai file del driver (10.0.1.8).
# Soluzione: pandas legge sul driver (dove il file esiste), poi createDataFrame
# distribuisce i dati agli executor tramite Arrow senza che questi tocchino disco.
DATA_PATH = f"{PATH}/all_elaborated.parquet"

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")  # conversione più veloce
print("Lettura dataset con pandas (driver-side)...")
_pdf = pd.read_parquet(DATA_PATH)
print(f"Pandas: {len(_pdf):,} righe, {len(_pdf.columns)} colonne — distribuzione agli executor...")
dataset = spark.createDataFrame(_pdf).repartition(32)
del _pdf  # libera RAM del driver non appena il piano Spark è stato creato
import gc as _gc; _gc.collect()

print(f"\nDataset distribuito: {dataset.count():,} righe, {len(dataset.columns)} colonne")

# ── Cast + pulizia infiniti in un unico select (evita query plan profondo) ─
# Ogni withColumn aggiunge un nodo al piano logico: 86 withColumn in loop
# possono causare StackOverflowError sulla JVM. Un singolo select è O(1).
_INF  = float("inf")
_NINF = float("-inf")

select_exprs = [F.col("Timestamp")]
for c in ALL_FEATURES:
    # Cast a double + sostituzione inf/-inf con null in un'unica espressione
    casted = F.col(c).cast(DoubleType())
    clean  = F.when(
        (casted == F.lit(_INF)) | (casted == F.lit(_NINF)), F.lit(None).cast(DoubleType())
    ).otherwise(casted).alias(c)
    select_exprs.append(clean)
select_exprs.append(F.col(LABEL_COL).cast(DoubleType()).alias(LABEL_COL))

df = dataset.select(*select_exprs).dropna(subset=[LABEL_COL])
print(f"Dopo pulizia: {len(ALL_FEATURES)} feature + Timestamp + {LABEL_COL}")

# ── Split temporale ───────────────────────────────────────────────────────
# Dataset: 2018-02-14 → 2018-03-02 (10 giornate di cattura)
# Split: primi 7 giorni per training (~70%), ultimi 3 per test (~30%)
SPLIT_DATE = "2018-02-24 00:00:00"

pt         = PuckTrick(df, engine=Engine.SPARK)
base_df    = pt.original

base_train_df = base_df.filter(F.col("Timestamp") < SPLIT_DATE).drop("Timestamp")
base_test_df  = base_df.filter(F.col("Timestamp") >= SPLIT_DATE).drop("Timestamp")

base_train_df.cache()
base_test_df.cache()

n_train        = base_train_df.count()
n_test         = base_test_df.count()
n_train_malign = base_train_df.filter(F.col(LABEL_COL) == 1).count()
n_test_malign  = base_test_df.filter(F.col(LABEL_COL) == 1).count()

print(f"\nTraining : {n_train:,} righe ({n_train_malign:,} malign, {n_train_malign/n_train*100:.2f}%)")
print(f"Test     : {n_test:,} righe ({n_test_malign:,} malign, {n_test_malign/n_test*100:.2f}%)")
print(f"Split    : {n_train/(n_train+n_test)*100:.1f}% train / {n_test/(n_train+n_test)*100:.1f}% test")

## Step 3 — Tuning iperparametri MLP sul dato pulito

### Perché il tuning è necessario

Il `MultilayerPerceptronClassifier` di Spark MLlib espone diversi iperparametri
che influenzano significativamente le performance del modello. Sceglierli in modo
arbitrario rischierebbe di introdurre un bias nei risultati: un modello mal
configurato potrebbe degradare non per il rumore introdotto, ma semplicemente
perché l'architettura è inadatta al problema.

Per questo motivo eseguiamo una sessione di tuning **una volta sola, sul dato
pulito**, e fissiamo gli iperparametri trovati per tutti gli esperimenti successivi.
Questo garantisce che qualsiasi variazione di F1-score osservata nei capitoli
successivi sia attribuibile **esclusivamente al rumore** e non a differenze di
configurazione del modello.

### Architettura del MLP

Il `MultilayerPerceptronClassifier` di Spark MLlib implementa un percettrone
multistrato con funzione di attivazione **sigmoide** per gli strati nascosti e
**softmax** per lo strato di output. Il parametro `layers` definisce la struttura
completa della rete come lista di interi, dove ogni elemento rappresenta il numero
di neuroni in quello strato.

Nel nostro caso:
- **Input layer**: dimensione fissa **43** (tutte le feature del CIC-IDS-2018 dopo preprocessing)
- **Hidden layer(s)**: da ottimizzare
- **Output layer**: dimensione fissa **2**, una per ciascuna classe
  (classe 0 = Benign, classe 1 = Malign)

### Iperparametri da ottimizzare

| Iperparametro | Descrizione | Valori testati |
|---|---|---|
| `layers` | Architettura completa della rete | `[43,64,2]`, `[43,128,2]`, `[43,128,64,2]`, `[43,128,64,32,2]` |
| `maxIter` | Numero massimo di iterazioni di ottimizzazione | `50`, `100` |
| `stepSize` | Learning rate dell'ottimizzatore L-BFGS | `0.01`, `0.05` |
| `blockSize` | Dimensione del blocco per aggregazione gradienti | `128`, `256` |

> **Nota su `blockSize`:** controlla quanti campioni vengono aggregati insieme
> prima di aggiornare i pesi. Valori più alti migliorano l'efficienza su cluster
> ma richiedono più memoria per executor. `128` è il default consigliato da Spark
> per dataset di questa dimensione.

> **Nota sull'input layer (43):** il dataset CIC-IDS-2018 ha 43 feature numeriche
> dopo il preprocessing (rimozione varianza zero + correlazione alta). Questo è
> significativamente più delle 13 feature del dataset MetroPT-3, quindi l'architettura
> della rete potrebbe richiedere strati nascosti più ampi.

### Strategia di ricerca

La ricerca avviene tramite **`CrossValidator`** con **3 fold** applicato su un
campione del **20% del training set** per ridurre i tempi di calcolo.

Il test set è **completamente escluso** dal tuning — viene usato solo alla fine
di questo step per calcolare la **baseline F1** sul dato pulito a 0% di rumore.

La metrica di selezione è **F1-score weighted**: tiene conto dello sbilanciamento
tra classi pesando il contributo di ciascuna classe proporzionalmente alla sua
frequenza.

### Output atteso

Al termine di questo step otteniamo:
- Le costanti `BEST_LAYERS`, `BEST_MAX_ITER`, `BEST_STEP` — fissate per tutti i run successivi
- La **baseline F1** sul test set pulito — il punto di riferimento per gli esperimenti

In [ ]:
# === STEP 3: Tuning iperparametri MLP sul dato pulito ===
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import PipelineModel
from pyspark.ml.classification import MultilayerPerceptronClassificationModel
import json

# ── Campione per il tuning (20% del training set) ─────────────────────────
tune_df, _ = base_train_df.randomSplit([0.2, 0.8], seed=42)
tune_df.cache()
print(f"Righe per tuning: {tune_df.count():,}")

# ── VectorAssembler ────────────────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols    = ALL_FEATURES,
    outputCol    = "features",
    handleInvalid= "keep"
)

# ── Modello base ───────────────────────────────────────────────────────────
mlp = MultilayerPerceptronClassifier(
    featuresCol = "features",
    labelCol    = LABEL_COL,
    seed        = 42
)

# ── Pipeline ───────────────────────────────────────────────────────────────
pipeline = Pipeline(stages=[assembler, mlp])

# ── Griglia iperparametri ──────────────────────────────────────────────────
# Input layer = N_FEATURES (43), Output layer = 2 (Benign/Malign)
param_grid = ParamGridBuilder() \
    .addGrid(mlp.layers, [
        [N_FEATURES, 64, 2],
        [N_FEATURES, 128, 2],
        [N_FEATURES, 128, 64, 2],
        [N_FEATURES, 128, 64, 32, 2]
    ]) \
    .addGrid(mlp.maxIter,    [50, 100]) \
    .addGrid(mlp.stepSize,   [0.01, 0.05]) \
    .addGrid(mlp.blockSize,  [128, 256]) \
    .build()

print(f"Combinazioni da testare: {len(param_grid)}")

# ── Evaluator ──────────────────────────────────────────────────────────────
evaluator = MulticlassClassificationEvaluator(
    labelCol      = LABEL_COL,
    predictionCol = "prediction",
    metricName    = "f1"
)

# ── CrossValidator ─────────────────────────────────────────────────────────
cv = CrossValidator(
    estimator          = pipeline,
    estimatorParamMaps = param_grid,
    evaluator          = evaluator,
    numFolds           = 3,
    seed               = 42,
    parallelism        = 4
)

# ── Esecuzione tuning ──────────────────────────────────────────────────────
print("Avvio tuning...")
t0 = time.time()
cv_model = cv.fit(tune_df)
print(f"Tuning completato in {(time.time()-t0)/60:.1f} minuti")

# ── Estrazione migliori iperparametri ──────────────────────────────────────
best_pipeline_model: PipelineModel = cv_model.bestModel  # type: ignore
best_mlp: MultilayerPerceptronClassificationModel = best_pipeline_model.stages[-1]  # type: ignore

BEST_LAYERS    = best_mlp.getLayers()
BEST_MAX_ITER  = best_mlp.getMaxIter()
BEST_STEP      = best_mlp.getStepSize()
BEST_BLOCK     = best_mlp.getBlockSize()

print("\n=== MIGLIORI IPERPARAMETRI ===")
print(f"  layers   : {BEST_LAYERS}")
print(f"  maxIter  : {BEST_MAX_ITER}")
print(f"  stepSize : {BEST_STEP}")
print(f"  blockSize: {BEST_BLOCK}")

# ── Baseline F1 sul test set pulito ───────────────────────────────────────
baseline_pred = best_pipeline_model.transform(base_test_df)
baseline_f1   = evaluator.evaluate(baseline_pred)

print(f"\n=== BASELINE (0% rumore) ===")
print(f"  F1-score sul test set pulito: {baseline_f1:.4f}")

# ── Salvataggio ────────────────────────────────────────────────────────────
PARAMS_PATH = f"{PATH_RESULTS}/mlp_best_params.json"
os.makedirs(os.path.dirname(PARAMS_PATH), exist_ok=True)

model_params = {
    "layers":      list(BEST_LAYERS),
    "maxIter":     BEST_MAX_ITER,
    "stepSize":    BEST_STEP,
    "blockSize":   BEST_BLOCK,
    "baseline_f1": baseline_f1
}

with open(PARAMS_PATH, "w") as f:
    json.dump(model_params, f, indent=2)

print(f"\n✅ Parametri salvati in: {PARAMS_PATH}")
print(json.dumps(model_params, indent=2))

In [ ]:
# ── Valori placeholder — da sostituire con l'output del tuning (Step 3) ────
# Decommentare ed eseguire Step 3 per ottenere i valori reali, poi aggiornarli qui.
# L'architettura [43, 128, 64, 2] è un punto di partenza ragionevole per
# 43 feature di input con classificazione binaria.
#BEST_LAYERS   = [N_FEATURES, 128, 64, 2]
#BEST_MAX_ITER = 100
#BEST_STEP     = 0.05
#BEST_BLOCK    = 128
#BASELINE_F1   = None  # verrà calcolato al primo run con 0% rumore

#print(f"Architettura MLP: {BEST_LAYERS}")
#print(f"maxIter: {BEST_MAX_ITER}, stepSize: {BEST_STEP}, blockSize: {BEST_BLOCK}")
#print(f"⚠️  Eseguire Step 3 (tuning) per ottenere i valori ottimali!")

---

## Step 4 — Funzione `inject_noise()`

### Obiettivo

In questo step definiamo la funzione `inject_noise()` — il wrapper attorno a
**Pucktrick** che verrà chiamato ad ogni run del loop sperimentale principale.

La funzione ha un compito preciso: dato il training set pulito, un tipo di rumore,
una feature target e una percentuale, restituisce un nuovo DataFrame con il rumore
iniettato secondo la strategia specificata. Il test set non viene mai toccato.

### Strategia di iniezione

Per ogni tipo di rumore, Pucktrick richiede una `strategy` — un dizionario che
specifica come e dove applicare la corruzione. I parametri chiave sono:

| Parametro | Descrizione |
|---|---|
| `affected_features` | Lista delle colonne su cui applicare il rumore |
| `selection_criteria` | Filtro sulle righe — `"all"` per applicare su tutto il dataset |
| `percentage` | Percentuale di righe da corrompere (es. `0.1` per 10%) |
| `mode` | Modalità di iniezione — `"new"` (indipendente, partiamo sempre da dati puliti) |
| `perturbate_data` | Configurazione della distribuzione di campionamento |

### I 5 tipi di rumore e il loro comportamento

**`duplicated`** — Duplica righe esistenti aggiungendole al DataFrame. Sul CIC-IDS-2018
questo potrebbe amplificare pattern di attacco specifici, causando overfitting.

**`labels`** — Inverte `Label_generic` (0↔1) su una percentuale di righe.
Particolarmente interessante con il forte sbilanciamento del dataset (~83% Benign):
il flip di etichette benigne a maligne crea molti falsi positivi nel training.

**`missing`** — Introduce valori `NaN` nelle colonne selezionate. I NaN vengono
gestiti dal `VectorAssembler` con `handleInvalid="keep"` (sostituzione con `0.0`).

**`noisy`** — Aggiunge rumore numerico sostituendo valori con casuali in `[min, max]`.
Le feature ad alta fragilità (`Bwd Pkts/s`, `Fwd Pkt Len Mean`) dovrebbero degradare
più rapidamente di `Fwd Seg Size Min` (robusta, categorica).

**`outliers`** — Introduce valori anomali oltre 3σ dalla media. Le feature con
alta kurtosis (come molte nel CIC-IDS-2018) sono particolarmente sensibili.

### Firma della funzione
```python
def inject_noise(
    pt,              # istanza PuckTrick già inizializzata
    train_df,        # DataFrame di training pulito
    noise_type,      # str: "duplicated"|"labels"|"missing"|"noisy"|"outliers"
    feature,         # str: feature su cui iniettare il rumore
    percentage,      # float: percentuale di righe da corrompere (0.0–0.5)
    seed             # int: seed per riproducibilità del campionamento
) -> DataFrame:      # DataFrame con rumore iniettato
```

In [ ]:
# === STEP 4: Funzione inject_noise() ===

def inject_noise(pt, train_df, noise_type, feature, percentage, seed):
    """
    Inietta rumore controllato su una feature del training set tramite Pucktrick.

    Args:
        pt          : istanza PuckTrick già inizializzata su base_train_df
        train_df    : DataFrame di training pulito (deve contenere ROW_ID)
        noise_type  : tipo di rumore da iniettare
        feature     : feature su cui applicare il rumore
        percentage  : percentuale di righe da corrompere (0.0 = nessun rumore)
        seed        : seed per riproducibilità del campionamento

    Returns:
        DataFrame con rumore iniettato, pronto per il training
    """

    # Baseline — nessun rumore
    if percentage == 0.0:
        return train_df

    # Strategia comune a tutti i tipi di rumore
    strategy = {
        "affected_features": [feature],
        "selection_criteria": "all",
        "percentage":         percentage,
        "mode":               "new",
        "perturbate_data": {
            "distribution": "random",
            "param":        {"seed": seed}
        }
    }

    # Dispatch per tipo di rumore
    if noise_type == "duplicated":
        status, noisy_df = pt.duplicated(train_df, strategy)
    elif noise_type == "labels":
        label_strategy = {**strategy, "affected_features": [LABEL_COL]}
        status, noisy_df = pt.labels(train_df, label_strategy)
    elif noise_type == "missing":
        status, noisy_df = pt.missing(train_df, strategy)
    elif noise_type == "noisy":
        status, noisy_df = pt.noise(train_df, strategy)
    elif noise_type == "outliers":
        status, noisy_df = pt.outlier(train_df, strategy)
    else:
        raise ValueError(f"Tipo di rumore non supportato: {noise_type}")

    if status != 0:
        print(f"⚠️  inject_noise: status={status} "
                f"({noise_type}, {feature}, {percentage:.0%}, seed={seed})")
        return train_df

    return noisy_df


# ── Inizializzazione PuckTrick sul training set ────────────────────────────
pt_train = PuckTrick(base_train_df, engine=Engine.SPARK)

# # ── Test rapido ────────────────────────────────────────────────────────────
# print("Test inject_noise()...")

# for noise_type in NOISE_TYPES:
#     test_noisy = inject_noise(
#         pt         = pt_train,
#         train_df   = pt_train.original,
#         noise_type = noise_type,
#         feature    = "Fwd Seg Size Min",
#         percentage = 0.1,
#         seed       = 42
#     )
#     n_orig  = pt_train.original.count()
#     n_noisy = test_noisy.count()
#     print(f"  ✅ {noise_type:<12} | righe: {n_orig:,} → {n_noisy:,}")

# print("\nTest completato.")

## Step 5 — Funzione `run_experiment()`

### Obiettivo

In questo step definiamo la funzione `run_experiment()` — il cuore del loop
sperimentale. Questa funzione esegue **un singolo run** dell'esperimento:
dato un tipo di rumore, una feature, una percentuale e un seed, restituisce
le metriche di performance del modello addestrato su dati rumorosi e valutato
sul test set pulito.

### Flusso di un singolo run
```
base_train_df (pulito — CIC-IDS-2018)
        ↓
inject_noise(noise_type, feature, percentage, seed)
        ↓
noisy_train_df
        ↓
VectorAssembler → features vector (43 feature)
        ↓
MultilayerPerceptronClassifier.fit()
        ↓
model.transform(base_test_df)
        ↓
F1-score + AUC-ROC
```

Il test set è **sempre pulito e fisso** — le metriche misurano esclusivamente
la capacità del modello addestrato su dati rumorosi di generalizzare su dati
reali non corrotti.

### Metriche calcolate

**F1-score weighted** — metrica principale. Tiene conto dello sbilanciamento
tra classi (Benign vs Malign) pesando proporzionalmente.

**AUC-ROC** — metrica di controllo. Misura la capacità discriminativa
indipendentemente dalla soglia di classificazione.

### Pipeline di training

Ad ogni run costruiamo una pipeline Spark MLlib con:

1. **`VectorAssembler`** — assembla le 43 feature in un unico vettore con
   `handleInvalid="keep"` per gestire i NaN
2. **`MultilayerPerceptronClassifier`** — con gli iperparametri fissi dal tuning

### Stima tempi

Con ~2.500 run totali e tempi variabili per run, il loop potrebbe richiedere
diverse ore. I risultati vengono salvati in modo incrementale su disco.

### Ottimizzazioni per il loop sperimentale

Per contenere i tempi del loop sperimentale (~2.500 run) sono state applicate
due ottimizzazioni alla funzione `run_experiment()`:

**Ottimizzazione 1 — Pipeline pre-costruita:** `VectorAssembler` e
`MultilayerPerceptronClassifier` vengono costruiti **una volta sola** fuori
dalla funzione e passati come parametri. Evita di ricreare gli oggetti pipeline
2.500 volte.

**Ottimizzazione 2 — `n_train` condizionale:** il conteggio delle righe del
training set dopo l'iniezione viene eseguito solo per `duplicated` — l'unico
tipo di rumore che modifica il numero di righe.

In [ ]:
# === STEP 5: Funzione run_experiment() — ottimizzata ===

# ── Oggetti pre-costruiti — creati una volta sola ─────────────────────────
base_assembler = VectorAssembler(
    inputCols    = ALL_FEATURES,
    outputCol    = "features",
    handleInvalid= "keep"
)

base_mlp = MultilayerPerceptronClassifier(
    featuresCol = "features",
    labelCol    = LABEL_COL,
    layers      = BEST_LAYERS,
    maxIter     = BEST_MAX_ITER,
    stepSize    = BEST_STEP,
    blockSize   = BEST_BLOCK
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol      = LABEL_COL,
    predictionCol = "prediction",
    metricName    = "f1"
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol         = LABEL_COL,
    rawPredictionCol = "rawPrediction",
    metricName       = "areaUnderROC"
)

N_TRAIN_BASE = pt_train.original.count()
print(f"✅ N_TRAIN_BASE: {N_TRAIN_BASE:,}")


def run_experiment(pt, noise_type, feature, percentage, seed):
    t0 = time.time()

    noisy_train = inject_noise(
        pt         = pt,
        train_df   = pt.original,
        noise_type = noise_type,
        feature    = feature,
        percentage = percentage,
        seed       = seed
    )

    mlp_run  = base_mlp.setSeed(seed * 100)
    pipeline = Pipeline(stages=[base_assembler, mlp_run])
    model    = pipeline.fit(noisy_train)

    predictions = model.transform(base_test_df)
    f1  = f1_evaluator.evaluate(predictions)
    auc = auc_evaluator.evaluate(predictions)

    duration = time.time() - t0
    n_train  = noisy_train.count() if noise_type == "duplicated" else N_TRAIN_BASE

    return {
        "noise_type": noise_type,
        "feature":    feature,
        "percentage": percentage,
        "seed":       seed,
        "f1":         f1,
        "auc":        auc,
        "n_train":    n_train,
        "duration_s": duration
    }

In [ ]:
# # === PROFILING TEMPI ===
# print("Profiling run_experiment()...")

# t0 = time.time()
# noisy_train = inject_noise(
#     pt         = pt_train,
#     train_df   = pt_train.original,
#     noise_type = "noisy",
#     feature    = "Fwd Seg Size Min",
#     percentage = 0.1,
#     seed       = 42
# )
# t1 = time.time()
# print(f"  inject_noise : {t1-t0:.1f}s")

# mlp_run  = base_mlp.setSeed(42 * 100)
# pipeline = Pipeline(stages=[base_assembler, mlp_run])
# model    = pipeline.fit(noisy_train)
# t2 = time.time()
# print(f"  pipeline.fit : {t2-t1:.1f}s")

# predictions = model.transform(base_test_df)
# f1  = f1_evaluator.evaluate(predictions)
# auc = auc_evaluator.evaluate(predictions)
# t3 = time.time()
# print(f"  evaluate     : {t3-t2:.1f}s")
# print(f"  totale       : {t3-t0:.1f}s")

## Step 6 — Loop sperimentale principale

### Struttura del loop

Il loop esegue tutte le combinazioni previste dal piano sperimentale:
```
Per ogni TIPO DI ERRORE (5):
    Se noise_type == "labels" o "duplicated":
        Per ogni PERCENTUALE (5):
            Per ogni RUN (20):
                run_experiment()
    Altrimenti:
        Per ogni FEATURE (3):
            Per ogni PERCENTUALE (5):
                Per ogni RUN (20):
                    run_experiment()
```

Totale run stimati: `(2 × 5 × 20) + (3 × 3 × 5 × 20) = 200 + 900 = 1.100 run`

### Salvataggio incrementale

I risultati vengono salvati su disco dopo ogni run in formato JSON lines.
In caso di interruzione, i run già completati non vengono persi.

### Ripresa automatica

All'avvio il loop verifica se esiste già un file di risultati parziali e
salta le combinazioni già eseguite — il loop è **idempotente**.

In [ ]:
# === STEP 6: Loop sperimentale principale ===
import json
import os
from datetime import datetime

RESULTS_PATH = f"{PATH_RESULTS}/mlp_cicids_results.jsonl"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# Tipi di rumore che non iterano sulle feature
NOISE_NO_FEATURE = ["labels", "duplicated"]

# ── Caricamento run già completati (resume) ───────────────────────────────
completed = set()
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                completed.add((r["noise_type"], r["feature"], r["percentage"], r["seed"]))
            except Exception:
                pass
    print(f"✅ Run già completati: {len(completed)}")
else:
    print("✅ Nessun risultato precedente — partenza da zero")

# ── Costruzione lista run da eseguire ─────────────────────────────────────
runs_to_do = []

for noise_type in NOISE_TYPES:
    if noise_type in NOISE_NO_FEATURE:
        # labels e duplicated non iterano sulle feature
        for percentage in NOISE_LEVELS:
            for seed in range(N_RUNS):
                key = (noise_type, "all_features", percentage, seed)
                if key not in completed:
                    runs_to_do.append((noise_type, "all_features", percentage, seed))
    else:
        for feature in NOISE_FEATURES:
            for percentage in NOISE_LEVELS:
                for seed in range(N_RUNS):
                    key = (noise_type, feature, percentage, seed)
                    if key not in completed:
                        runs_to_do.append((noise_type, feature, percentage, seed))

total_runs = (len(NOISE_NO_FEATURE) * len(NOISE_LEVELS) * N_RUNS) + \
             ((len(NOISE_TYPES) - len(NOISE_NO_FEATURE)) * len(NOISE_FEATURES) * len(NOISE_LEVELS) * N_RUNS)
remaining  = len(runs_to_do)

print(f"✅ Run totali    : {total_runs}")
print(f"✅ Run rimanenti : {remaining}")
print(f"✅ Stima tempo   : {remaining * 60 / 3600:.1f} ore (@ 60s/run)")
print(f"✅ Avvio         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ── Loop principale ────────────────────────────────────────────────────────
with open(RESULTS_PATH, "a") as results_file:
    for i, (noise_type, feature, percentage, seed) in enumerate(runs_to_do):

        try:
            # Per labels e duplicated passa la prima NOISE_FEATURE — viene ignorata
            feature_arg = NOISE_FEATURES[0] if noise_type in NOISE_NO_FEATURE else feature

            result = run_experiment(
                pt         = pt_train,
                noise_type = noise_type,
                feature    = feature_arg,
                percentage = percentage,
                seed       = seed
            )

            # Sovrascrivi feature con il valore corretto per il salvataggio
            result["feature"]   = feature
            result["timestamp"] = datetime.now().isoformat()

            # Salvataggio incrementale
            results_file.write(json.dumps(result) + "\n")
            results_file.flush()

            # Log ogni 10 run
            if (i + 1) % 10 == 0:
                avg_duration = result["duration_s"]
                elapsed_h    = (i + 1) * avg_duration / 3600
                remaining_h  = (remaining - i - 1) * avg_duration / 3600
                print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                    f"Run {i+1}/{remaining} | "
                    f"{noise_type:<12} | {feature:<25} | "
                    f"{percentage:.0%} | seed={seed} | "
                    f"F1={result['f1']:.4f} | "
                    f"elapsed≈{elapsed_h:.1f}h | "
                    f"remaining≈{remaining_h:.1f}h")

        except Exception as e:
            print(f"❌ ERRORE run ({noise_type}, {feature}, {percentage}, seed={seed}): {e}")
            continue

print(f"\n✅ Loop completato — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✅ Risultati salvati in: {RESULTS_PATH}")